# Tokenization

In this notebook, we will build the first component needed to train a standard transformer language model from scratch, the tokenizer.

**What we will do:**
1. Implement a byte-pair encoding (BPE) tokenizer.
2. Train a BPE tokenizer on the TinyStories dataset.
3. Run this trained tokenizer on the dataset to convert it into a sequence of integer IDs.

**Datasets:**
We will need the following two datasets in this notebook:
- TinyStories.
- OpenWebText (sample).

We will train and implement a byte-level byte-pair encoding (BPE) tokenizer. In particular, we will represent arbitrary (Unicode) strings as a sequence of bytes and train our BPE tokenizer on this byte sequence. Later, we will use this tokenizer to encode text (a string) into tokens (a sequence of integers) for language modeling.

In [ ]:
import re
import regex
import numpy as np

import sys; sys.path.insert(0, '..')
from tests.test_tokenization import *

In [ ]:
seed = 42
np.random.seed(seed)

## The Unicode Standard

Unicode is a text encoding standard that maps characters to integer code points. As of Unicode 16.0 (released in September 2024), the standard defines 154,998 characters across 168 scripts. For example, the character "s" has the code point 115 (typically notated as U+0073, where U+ is a conventional prefix and 0073 is 115 in hexadecimal), and the character "牛" has the code point 29275. In Python, you can use the `ord()` function to convert a single Unicode character into its integer representation. The `chr()` function converts an integer Unicode code point into a string with the corresponding character.

```python
>>> ord('牛')
29275
>>> chr(29275)
'牛'
```

### Problem: Understanding Unicode

Provide a one-sentence explanation for each question.
1. What Unicode character does `chr(0)` return?
2. How does this character's string representation (`__repr__()`) differ from its printed representation?
3. What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:

```python
>>> chr(0)
>>> print(chr(0))
>>> 'this is a test' + chr(0) + 'string'
>>> print('this is a test' + chr(0) + 'string')
```

In [ ]:
# answers

## Unicode Encodings

While the Unicode standard defines a mapping from characters to code points (integers), it's impractical to train tokenizers directly on Unicode codepoints, since the vocabulary would be prohibitively large (around 150K items) and sparse (since many characters are quite rare). Instead, we'll use a Unicode encoding, which converts a Unicode character into a sequence of bytes. The Unicode standard itself defines three encodings: UTF-8, UTF-16, and UTF-32, with UTF-8 being the dominant encoding for the Internet (more than 98% of all webpages).

To encode a Unicode string into UTF-8, we can use the `encode()` function in Python. To access the underlying byte values for a Python bytes object, we can iterate over it, e.g. with `list()`. Finally, we can use the `decode()` function to decode a UTF-8 byte string into a Unicode string.

```python
>>> test_string = 'hello! こんにちは!'
>>> utf8_encoded = test_string.encode('utf-8')
>>> print(utf8_encoded)
b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
>>> print(type(utf8_encoded))
<class 'bytes'>
>>> # Get the byte values for the encoded string (integers from 0 to 255).
>>> list(utf8_encoded)
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129,
161, 227, 129, 175, 33]
>>> # One byte does not necessarily correspond to one Unicode character!
>>> print(len(test_string))
13
>>> print(len(utf8_encoded))
23
>>> print(utf8_encoded.decode('utf-8'))
hello! こんにちは!
```

By converting our Unicode codepoints into a sequence of bytes (e.g. via the UTF-8 encoding), we are essentially taking a sequence of codepoints (integers in the range 0 to 154,997) and transforming it into a sequence of byte values (integers in the range 0 to 255). The 256-length byte vocabulary is much more manageable to deal with. When using byte-level tokenization, we do not need to worry about out-of vocabulary tokens, since we know that any input text can be expressed as a sequence of integers from 0 to 255.

### Problem: Unicode Encodings

1. What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.
2. Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.

```python
def decode_utf8_bytes_to_str_wrong(bytestring):
    return ''.join([bytes([b]).decode('utf-8') for b in bytestring])

>>> decode_utf8_bytes_to_str_wrong('hello'.encode('utf-8'))
'hello'
```
3. Give a two byte sequence that does not decode to any Unicode character(s).

In [ ]:
# answers

## Subword Tokenization

While byte-level tokenization can alleviate the out-of-vocabulary issues faced by word-level tokenizers, tokenizing text into bytes results in extremely long input sequences. This slows down model training, since a sentence with 10 words might only be 10 tokens long in a word-level language model, but could be 50 or more tokens long in a character-level model (depending on the length of the words). Processing these longer sequences requires more computation at each step of the model. Furthermore, language modeling on byte sequences is difficult because the longer input sequences create long-term dependencies in the data. 

Subword tokenization is a midpoint between word-level tokenizers and byte-level tokenizers. Note that a byte-level tokenizer's vocabulary has 256 entries (byte values are 0 to 225). A subword tokenizer trades-off a larger vocabulary size for better compression of the input byte sequence. For example, if the byte sequence `b'the'` often occurs in our raw text training data, assigning it an entry in the vocabulary would reduce this 3-token sequence to a single token.

How do we select these subword units to add to our vocabulary? *Sennrich et al. (2016)* propose to use byte-pair encoding (BPE; *Gage, 1994*), a compression algorithm that iteratively replaces ("merges") the most frequent pair of bytes with a single, new unused index. Note that this algorithm adds subword tokens to our vocabulary to maximize the compression of our input sequences—if a word occurs in our input text
enough times, it'll be represented as a single subword unit.

Subword tokenizers with vocabularies constructed via BPE are often called BPE tokenizers. In this problem, we'll implement a byte-level BPE tokenizer, where the vocabulary items are bytes or merged sequences of bytes, which give us the best of both worlds in terms of out-of-vocabulary handling and manageable input sequence lengths. The process of constructing the BPE tokenizer vocabulary is known as "training" the BPE tokenizer.

## BPE Tokenizer Training

The BPE tokenizer training procedure consists of three main steps.

**Vocabulary initialization:** The tokenizer vocabulary is a one-to-one mapping from bytestring token to integer ID. Since we're training a byte-level BPE tokenizer, our initial vocabulary is simply the set of all bytes. Since there are 256 possible byte values, our initial vocabulary is of size 256.

**Pre-tokenization:** Once you have a vocabulary, you could, in principle, count how often bytes occur next to each other in your text and begin merging them starting with the most frequent pair of bytes. However, this is quite computationally expensive, since we'd have to go take a full pass over the corpus each time we merge. In addition, directly merging bytes across the corpus may result in tokens that differ only in punctuation (e.g. `dog!` vs. `dog.`). These tokens would get completely different token IDs, even though they are likely to have high semantic similarity (since they differ only in punctuation).

To avoid this, we pre-tokenize the corpus. You can think of this as a coarse-grained tokenization over the corpus that helps us count how often pairs of characters appear. For example, the word `text` might be a pre-token that appears 10 times. In this case, when we count how often the characters `t` and `e` appear next to each other, we will see that the word `text` has `t` and `e` adjacent and we can increment their count by 10 instead of looking through the corpus. Since we're training a byte-level BPE model, each pre-token is represented as a sequence of UTF-8 bytes.

The original BPE implementation of *Sennrich et al. (2016)* pre-tokenizes by simply splitting on whitespace (i.e. `s.split(" ")`). In contrast, we'll use a regex-based pre-tokenizer (used by GPT-2; *Radford et al., 2019*) from [tiktoken](https://github.com/openai/tiktoken/pull/234/files):

```python
>>> PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

It may be useful to interactively split some text with this pre-tokenizer to get a better sense of its behavior:

```python
>>> # requires `regex` package
>>> import regex as re
>>> re.findall(PAT, 'some text that i'll pre-tokenize')
['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']
```

When using it in your code, however, you should use `re.finditer` to avoid storing the pre-tokenized words as you construct your mapping from pre-tokens to their counts.

**Compute BPE merges:** Now that we've converted our input text into pre-tokens and represented each pre-token as a sequence of UTF-8 bytes, we can compute the BPE merges (i.e. train the BPE tokenizer). At a high level, the BPE algorithm iteratively counts every pair of bytes and identifies the pair with the highest frequency (`A`, `B`). Every occurrence of this most frequent pair (`A`, `B`) is then merged, i.e. replaced with a new token `AB`. This new merged token is added to our vocabulary; as a result, the final vocabulary after BPE training is the size of the initial vocabulary (256 in our case), plus the number of BPE merge operations performed during training. For efficiency during BPE training, we do not consider pairs that cross pre-token boundaries. When computing merges, deterministically break ties in pair frequency by preferring the lexicographically greater pair. For example, if the pairs (`A`, `B`), (`A`, `C`), (`B`, `ZZ`), and (`BA`, `A`) all have the highest frequency, we'd merge (`BA`, `A`):

```python
>>> max([('A', 'B'), ('A', 'C'), ('B', 'ZZ'), ('BA', 'A')])
('BA', 'A')
```

*Note:* The original BPE formulation (*Sennrich et al., 2016)* specifies the inclusion of an end-of-word token. We do not add an end-of-word-token when training byte-level BPE models because all bytes (including whitespace and punctuation) are included in the model’s vocabulary. Since we’re explicitly representing spaces and punctuation, the learned BPE merges will naturally reflect these word boundaries.

**Special tokens:** Often, some strings (e.g. `<|endoftext|>`) are used to encode metadata (e.g. boundaries between documents). When encoding text, it's often desirable to treat some strings as "special tokens" that should never be split into multiple tokens (i.e. will always be preserved as a single token). For example, the end-of-sequence string `<|endoftext|>` should always be preserved as a single token (i.e. a single integer ID), so we know when to stop generating from the language model. These special tokens must be added to the vocabulary, so they have a corresponding fixed token ID.

*Algorithm 1 of Sennrich et al. (2016)* contains an inefficient implementation of BPE tokenizer training (essentially following the steps that we outlined above). As a first exercise, it may be useful to implement and test this function to test your understanding.

### Example: BPE training example

Here is a stylized example from *Sennrich et al. (2016)*. Consider a corpus consisting of the following text:

```
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
```

and the vocabulary has a special token `<|endoftext|>`.

*Vocabulary:* We initialize our vocabulary with our special token `<|endoftext|>` and the 256 byte values.

*Pre-tokenization:* For simplicity and to focus on the merge procedure, we assume in this example that pretokenization simply splits on whitespace. When we pretokenize and count, we end up with the frequency table.

```
{low: 5, lower: 2, widest: 3, newest: 6}
```

It is convenient to represent this as a `dict[tuple[bytes], int]`, e.g. `{(l,o,w): 5}`. Note that even a single byte is a bytes object in Python. There is no byte type in Python to represent a single byte, just as there is no char type in Python to represent a single character.

*Merges:* We first look at every successive pair of bytes and sum the frequency of the words where they appear `{lo: 7, ow: 7, we: 8, er: 2, wi: 3, id: 3, de: 3, es: 9, st: 9, ne: 6, ew: 6}`. The pair (`es`) and (`st`) are tied, so we take the lexicographically greater pair, (`st`). We would then merge the pre-tokens so that we end up with `{(l,o,w): 5, (l,o,w,e,r): 2, (w,i,d,e,st): 3, (n,e,w,e,st): 6}`.

In the second round, we see that (`e`, `st`) is the most common pair (with a count of 9) and we would merge into `{(l,o,w): 5, (l,o,w,e,r): 2, (w,i,d,est): 3, (n,e,w,est): 6}`. Continuing this, the sequence of merges we get in the end will be `['s t', 'e st', 'o w', 'l ow', 'w est', 'n e', 'ne west', 'w i', 'wi d', 'wid est', 'low e', 'lowe r']`.

If we take 6 merges, we have `['s t', 'e st', 'o w', 'l ow', 'w est', 'n e']` and our vocabulary elements would be `[<|endoftext|>, [...256 BYTE CHARS], st, est, ow, low, west, ne]`.

With this vocabulary and set of merges, the word newest would tokenize as `[ne, west]`.

## Experimenting with BPE Tokenizer Training

Let's train a byte-level BPE tokenizer on the TinyStories dataset. You can download these datasets wherever you like with the following:

``` sh
wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
wget https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt
```

Before you start, we recommend taking a look at the TinyStories dataset to get a sense of what's in the data.

**Parallelizing pre-tokenization:** we will find that a major bottleneck is the pre-tokenization step. You can speed up pre-tokenization by parallelizing your code with the built-in library multiprocessing. Concretely, we recommend that in parallel implementations of pre tokenization, you chunk the corpus while ensuring your chunk boundaries occur at the beginning of a special token. You are free to use the starter code in `refs/pretokenization_example.py` verbatim to obtain chunk boundaries, which you can then use to distribute work across your processes.

This chunking will always be valid, since we never want to merge across document boundaries. For the purposes of this notebook, you can always split in this way. Don't worry about the edge case of receiving a very large corpus that does not contain `<|endoftext|>`.

**Removing special tokens before pre-tokenization:** Before running pre-tokenization with the regex pattern (using `re.finditer`), you should strip out all special tokens from your corpus (or your chunk, if using a parallel implementation). Make sure that you *split* on your special tokens, so that no merging can occur across the text they delimit. For example, if you have a corpus (or chunk) like `[Doc 1]<|endoftext|>[Doc 2]`, you should split on the special token `<|endoftext|>`, and pre-tokenize `[Doc 1]` and `[Doc 2]` separately, so that no merging can occur across the document boundary. This can be done using `re.split` with `'|'.join(special_tokens)` as the delimiter (with careful use of `re.escape` since `|` may occur in the special tokens). The test `test_train_bpe_special_tokens` will test for this.

**Optimizing the merging step:** The naïve implementation of BPE training in the stylized example above is slow because for every merge, it iterates over all byte pairs to identify the most frequent pair. However, the only pair counts that change after each merge are those that overlap with the merged pair. Thus, BPE training speed can be improved by indexing the counts of all pairs and incrementally updating these
counts, rather than explicitly iterating over each pair of bytes to count pair frequencies. You can get significant speedups with this caching procedure, though we note that the merging part of BPE training is not parallelizable in Python.

*Tip (Profiling):* Consider using profiling tools like `cProfile` or `scalene` to identify the bottlenecks in your implementation, and focus on optimizing those.

*Tip (Downscaling):* Instead of jumping to training your tokenizer on the full TinyStories dataset, we recommend you first train on a small subset of the data: a "debug dataset". For example, you could train your tokenizer on the TinyStories validation set instead, which is 22K documents instead of 2.12M. This illustrates a general strategy of downscaling whenever possible to speed up development: for example, using smaller datasets, smaller model sizes, etc. Choosing the size of the debug dataset or hyperparameter config requires careful consideration: you want your debug set to be large enough to have the same bottlenecks as the full configuration (so that the optimizations you make will generalize), but not so big that it takes forever to run.

### Problem: BPE Tokenizer Training

Fill in the function `train_bpe` below. This function should take in a given path to an input text file, train a (byte-level) BPE tokenizer, and return the vocabulary dictionary and list of merge pairs.

To test your BPE training function we will use the `run_train_bpe` adapter filled in below. Try to make sure all tests are passing before proceeding to the next section.

*Optional:* With a larger time-investment, you can implement the key parts of your training method using some systems language, for instance C++ (consider `cppyy` for this) or Rust (using `PyO3`). If you do this, be aware of which operations require copying vs reading directly from Python memory, and make sure to leave build instructions. Also note that the GPT-2 regex is not well-supported in most regex engines and will be too slow in most that do. We have verified that Oniguruma is reasonably fast and supports negative lookahead, but the regex package in Python is, if anything, even faster.

In [ ]:
def train_bpe(input_path, vocab_size, special_tokens):
    """
    Given the path to an input corpus, train a BPE tokenizer and output its vocabulary and merges.
    Args:
        input_path (str | os.PathLike): Path to BPE tokenizer training data.
        vocab_size (int): Total number of items in the tokenizer's vocabulary (including special tokens).
        special_tokens (list[str]): A list of string special tokens to be added to the tokenizer vocabulary.
            These strings will never be split into multiple tokens, and will always be kept as a single token. 
            If these special tokens occur in the `input_path`, they are treated as any other string.
    Returns:
        tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
            vocab:
                The trained tokenizer vocabulary, a mapping from int (token ID in the vocabulary) to bytes (token bytes)
            merges:
                BPE merges. Each list item is a tuple of bytes (<token1>, <token2>), representing that <token1> was merged with <token2>.
                Merges are ordered by order of creation.
    """
    raise NotImplementedError

In [ ]:
def run_train_bpe(input_path, vocab_size, special_tokens):
    """
    Adapter function for the main train_bpe function.
    """
    return train_bpe(input_path, vocab_size, special_tokens)

test_train_bpe_speed(run_train_bpe)
test_train_bpe(run_train_bpe)
test_train_bpe_special_tokens(run_train_bpe)

### Problem: BPE Training on TinyStories

Provide a one to two sentence response to each question.
1. Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary size of 10,000. Make sure to add the TinyStories `<|endoftext|>` special token to the vocabulary. Serialize the resulting vocabulary and merges to disk for further inspection. How many hours and memory did training take? What is the longest token in the vocabulary? Does it make sense?
    - Try to have this run in under 30 minutes on a CPU and use less than 30GB of RAM.
    - *Hint:*  You should be able to get under 2 minutes for BPE training using multiprocessing during pretokenization and the following two facts about the `<|endoftext|>` token: (1) It delimits documents in the data files. (2) It's handled as a special case before the BPE merges are applied.
2. Profile your code. What part of the tokenizer training process takes the most time?

In [ ]:
# implementation code

Next, we'll try training a byte-level BPE tokenizer on the OpenWebText dataset. As before, we recommend taking a look at the dataset to better understand its contents. You can download and unzip this data wherever you like with the following shell commands:

```sh
wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_train.txt.gz
gunzip owt_train.txt.gz
wget https://huggingface.co/datasets/stanford-cs336/owt-sample/resolve/main/owt_valid.txt.gz
gunzip owt_valid.txt.gz
```

### Problem: BPE Training on OpenWebText

Provide a one to two sentence response to each question.
1. Train a byte-level BPE tokenizer on the OpenWebText dataset, using a maximum vocabulary size of 32,000. Serialize the resulting vocabulary and merges to disk for further inspection. What is the longest token in the vocabulary? Does it make sense?
    - Try to have this run in under 12 hours on the CPU, and use less than 100GB RAM.
3. Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.

In [ ]:
# implementation code

## BPE Tokenizer: Encoding and Decoding

In the previous section, we implemented a function to train a BPE tokenizer on input text to obtain a tokenizer vocabulary and a list of BPE merges. Now, we will implement a BPE tokenizer that loads a provided vocabulary and list of merges and uses them to encode and decode text to/from token IDs.

### Encoding text
The process of encoding text by BPE mirrors how we train the BPE vocabulary. There are a few major steps.
1. *Pre-tokenize:* We first pre-tokenize the sequence and represent each pre-token as a sequence of UTF-8 bytes, just as we did in BPE training. We will be merging these bytes within each pre-token into vocabulary elements, handling each pre-token independently (no merges across pre-token boundaries).
2. *Apply the merges:* We then take the sequence of vocabulary element merges created during BPE training, and apply it to our pre-tokens *in the same order of creation*.

**Example: BPE encoding example**

For example, suppose our input string is 'the cat ate', our vocabulary is `{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b'at'}`, and our learned merges are `[(b't', b'h'), (b' ', b'c'), (b' ', 'a'), (b'th', b'e'), (b' a', b't')]`. First, our pre-tokenizer would split this string into `['the', ' cat', ' ate']`. Then, we'll look at each pre-token and apply the BPE merges.

The first pre-token 'the' is initially represented as `[b't', b'h', b'e']`. Looking at our list of merges, we identify the first applicable merge to be (`b't'`, `b'h'`), and use that to transform the pre-token into `[b'th', b'e']`. Then, we go back to the list of merges and identify the next applicable merge to be (`b'th'`, `b'e'`), which transforms the pre-token into `[b'the']`. Finally, looking back at
the list of merges, we see that there are no more that apply to the string (since the entire pre-token has been merged into a single token), so we are done applying the BPE merges. The corresponding integer sequence is `[9]`.

Repeating this process for the remaining pre-tokens, we see that the pre-token `' cat'` is represented as `[b' c', b'a', b't']` after applying the BPE merges, which becomes the integer sequence `[7, 1, 5]`. The final pre-token `' ate'` is `[b' at', b'e']` after applying the BPE merges, which becomes the integer sequence `[10, 3]`. Thus, the final result of encoding our input string is `[9, 7, 1, 5, 10, 3]`.

*Special tokens:* Your tokenizer should be able to properly handle user-defined special tokens when encoding text (provided when constructing the tokenizer).

*Memory considerations:* Suppose we want to tokenize a large text file that we cannot fit in memory. To efficiently tokenize this large file (or any other stream of data), we need to break it up into manageable chunks and process each chunk in-turn, so that the memory complexity is constant as opposed to linear in the size of the text. In doing so, we need to make sure that a token doesn't cross chunk boundaries, else we'll get a different tokenization than the naïve method of tokenizing the entire sequence in-memory.

### Decoding text

To decode a sequence of integer token IDs back to raw text, we can simply look up each ID's corresponding entries in the vocabulary (a byte sequence), concatenate them together, and then decode the bytes to a Unicode string. Note that input IDs are not guaranteed to map to valid Unicode strings (since a user could input any sequence of integer IDs). In the case that the input token IDs do not produce a valid
Unicode string, you should replace the malformed bytes with the official Unicode replacement character `U+FFFD`. The errors argument of `bytes.decode` controls how Unicode decoding errors are handled, and using errors='replace' will automatically replace malformed data with the replacement marker.

### Problem: Implementing the tokenizer

Implement a `Tokenizer` class that, given a vocabulary and a list of merges, encodes text into integer IDs and decodes integer IDs into text. Your tokenizer should also support user-provided special tokens (appending them to the vocabulary if they aren't already there). Follow the interface shown below when implementing this class.

To test your Tokenizer against our provided tests, we will use the `Tokenizer` class you defined to implement the test adapter function `get_tokenizer`. This adapter will then be used to execute a series of tests of your `Tokenizer` class. Try to make sure all tests are passing before proceeding.

In [ ]:
class Tokenizer():
    def __init__(self, vocab, merges, special_tokens=None):
        """
        Construct a tokenizer from a given vocabulary, list of merges, and (optionally) a list of special tokens. 
        Args:
            vocab: dict[int, bytes]
            merges: list[tuple[bytes, bytes]]
            special_tokens: list[str] | None = None
        """
        raise NotImplementedError
    
    @classmethod
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):
        """
        Class method that constructs and returns a Tokenizer from a serialized vocabulary and list of merges 
        (in the same format that your BPE training code output) and (optionally) a list of special tokens.
        Args:
            vocab_filepath: str
            merges_filepath: str
            special_tokens: list[str] | None = None
        Returns:
            Tokenizer
        """
        raise NotImplementedError
    
    def encode(self, text):
        """
        Encode an input text into a sequence of token IDs.
        Args:
            text: str
        Returns:
            ids: list[int]
        """
        raise NotImplementedError
    
    def encode_iterable(self, iterable):
        """
        Given an iterable of strings (e.g. a Python file handle), return a generator that lazily yields token IDs. 
        This is required for memory-efficient tokenization of large files that we cannot directly load into memory.
        Args:
            iterable: Iterable[str]
        Returns:
            ids: Iterator[int]
        """
        raise NotImplementedError
    
    def decode(self, ids):
        """
        Decode a sequence of token IDs into text.
        Args:
            ids: list[int]
        Returns:
            text: str
        """
        raise NotImplementedError

In [ ]:
def get_tokenizer(vocab, merges, special_tokens):
    """
    Adapter function for the Tokenizer class.
    """
    return Tokenizer(vocab, merges, special_tokens)

test_roundtrip_empty(get_tokenizer)
test_empty_matches_tiktoken(get_tokenizer)
test_roundtrip_single_character(get_tokenizer)
test_single_character_matches_tiktoken(get_tokenizer)
test_roundtrip_single_unicode_character(get_tokenizer)
test_single_unicode_character_matches_tiktoken(get_tokenizer)
test_roundtrip_ascii_string(get_tokenizer)
test_ascii_string_matches_tiktoken(get_tokenizer)
test_roundtrip_unicode_string(get_tokenizer)
test_unicode_string_matches_tiktoken(get_tokenizer)
test_roundtrip_unicode_string_with_special_tokens(get_tokenizer)
test_unicode_string_with_special_tokens_matches_tiktoken(get_tokenizer)
test_overlapping_special_tokens(get_tokenizer)
test_address_roundtrip(get_tokenizer)
test_address_matches_tiktoken(get_tokenizer)
test_german_roundtrip(get_tokenizer)
test_german_matches_tiktoken(get_tokenizer)
test_tinystories_sample_roundtrip(get_tokenizer)
test_tinystories_matches_tiktoken(get_tokenizer)
test_encode_special_token_trailing_newlines(get_tokenizer)
test_encode_special_token_double_newline_non_whitespace(get_tokenizer)
test_encode_iterable_tinystories_sample_roundtrip(get_tokenizer)
test_encode_iterable_tinystories_matches_tiktoken(get_tokenizer)

## Experiments

Provide a one to two sentence response to each question.

1. Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively), encode these sampled documents into integer IDs. What is each tokenizer's compression ratio (bytes/token)?
2. What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer? Compare the compression ratio and/or qualitatively describe what happens.
3. Estimate the throughput of your tokenizer (e.g. in bytes/second). How long would it take to tokenize the Pile dataset (825GB of text)?
4. Using your TinyStories and OpenWebText tokenizers, encode the respective training and development datasets into a sequence of integer token IDs. We'll use this later to train our language model. We recommend serializing the token IDs as a NumPy array of datatype `uint16`. Why is uint16 an appropriate choice?

In [ ]:
# implementation code